# 04 - Vietnamese ASR (all videos)

Process all MP4 files, keep Faster-Whisper on GPU, save per-video output, and print transcript samples after each video.


In [ ]:
# Pin CTranslate2: newer wheels may require a CUDA runtime newer than Kaggle P100/T4 drivers.
%pip install -q --upgrade "faster-whisper==1.1.0" "ctranslate2==4.4.0" pandas pyarrow
!command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -qq ffmpeg)


In [ ]:
from __future__ import annotations

import gc
import json
import shutil
import traceback
from pathlib import Path

import pandas as pd
from IPython.display import display
from faster_whisper import WhisperModel

PIPELINE_VERSION = "aicv3-asr-vi"
BATCH_NOTEBOOK_VERSION = "2026-08-18-asr-all-videos-v1"
MODEL_NAME = "large-v3"
LANGUAGE = "vi"
BEAM_SIZE = 5
CHECKPOINT_EVERY = 25
PREVIEW_SEGMENTS = 12

INPUT_ROOT = Path("/kaggle/input")
BATCH_ROOT = Path("/kaggle/working/04-asr-vietnamese-all-videos-output")
BATCH_ROOT.mkdir(parents=True, exist_ok=True)
VIDEO_PATHS = sorted(INPUT_ROOT.rglob("*.mp4"))
if not VIDEO_PATHS:
    raise FileNotFoundError("No MP4 video found in /kaggle/input")
print("Batch notebook version:", BATCH_NOTEBOOK_VERSION)
print("Videos found:", len(VIDEO_PATHS))


In [ ]:
# Load Faster-Whisper only once for the complete batch.
# int8_float16 lowers VRAM usage while keeping GPU inference and good ASR quality.
try:
    model = WhisperModel(MODEL_NAME, device="cuda", compute_type="int8_float16")
except RuntimeError as exc:
    raise RuntimeError(
        "Faster-Whisper could not initialize CUDA. Restart the Kaggle session after "
        "running the install cell, confirm GPU is enabled, then run all cells again. "
        f"Original error: {exc}"
    ) from exc
print("Whisper model ready:", MODEL_NAME)


In [ ]:
success_count = 0
failure_count = 0
failures = []

for video_number, VIDEO_PATH in enumerate(VIDEO_PATHS, start=1):
    VIDEO_ID = VIDEO_PATH.stem
    OUTPUT_ROOT = BATCH_ROOT / VIDEO_ID
    CHECKPOINT_PATH = OUTPUT_ROOT / "asr-checkpoint.jsonl"
    SUCCESS_PATH = OUTPUT_ROOT / "_SUCCESS.json"
    ERROR_PATH = OUTPUT_ROOT / "_ERROR.txt"

    if SUCCESS_PATH.exists():
        print(f"[{video_number}/{len(VIDEO_PATHS)}] Skip completed {VIDEO_ID}")
        success_count += 1
        continue

    print(f"\n[{video_number}/{len(VIDEO_PATHS)}] Processing {VIDEO_ID}: {VIDEO_PATH}")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    records = []

    try:
        segments_generator, info = model.transcribe(
            str(VIDEO_PATH),
            language=LANGUAGE,
            beam_size=BEAM_SIZE,
            vad_filter=True,
            vad_parameters={"min_silence_duration_ms": 500, "speech_pad_ms": 250},
            condition_on_previous_text=True,
            word_timestamps=True,
        )

        with CHECKPOINT_PATH.open("w", encoding="utf-8") as checkpoint:
            for segment_index, segment in enumerate(segments_generator):
                words = [
                    {
                        "start_time": float(word.start),
                        "end_time": float(word.end),
                        "word": word.word.strip(),
                        "probability": float(word.probability),
                    }
                    for word in (segment.words or [])
                ]
                record = {
                    "segment_id": f"{VIDEO_ID}:asr:{segment_index:06d}",
                    "video_id": VIDEO_ID,
                    "segment_index": segment_index,
                    "start_time": float(segment.start),
                    "end_time": float(segment.end),
                    "text": segment.text.strip(),
                    "language": LANGUAGE,
                    "avg_logprob": float(segment.avg_logprob),
                    "no_speech_prob": float(segment.no_speech_prob),
                    "words": words,
                }
                records.append(record)
                checkpoint.write(json.dumps(record, ensure_ascii=False) + "\n")
                checkpoint.flush()
                if segment_index == 0 or (segment_index + 1) % CHECKPOINT_EVERY == 0:
                    print(f"ASR {segment_index + 1}: {segment.start:.2f}s-{segment.end:.2f}s | {segment.text.strip()}")

        if not records:
            raise RuntimeError("No speech segment was recognized")

        asr_df = pd.DataFrame(records).sort_values("start_time").reset_index(drop=True)
        assert asr_df["segment_id"].is_unique
        assert (asr_df["start_time"] >= 0).all()
        assert (asr_df["end_time"] > asr_df["start_time"]).all()
        assert asr_df["start_time"].is_monotonic_increasing
        assert asr_df["end_time"].max() <= info.duration + 1.0

        asr_df.to_json(OUTPUT_ROOT / "asr-segments.jsonl", orient="records", lines=True, force_ascii=False)
        parquet_df = asr_df.copy()
        parquet_df["words_json"] = parquet_df["words"].map(lambda value: json.dumps(value, ensure_ascii=False))
        parquet_df.drop(columns=["words"]).to_parquet(OUTPUT_ROOT / "asr-segments.parquet", index=False)
        (OUTPUT_ROOT / "transcript.txt").write_text(" ".join(asr_df["text"]), encoding="utf-8")

        summary = {
            "stage": "asr",
            "pipeline_version": PIPELINE_VERSION,
            "video_id": VIDEO_ID,
            "model": f"faster-whisper/{MODEL_NAME}",
            "language": LANGUAGE,
            "language_probability": float(info.language_probability),
            "duration_sec": float(info.duration),
            "segments": int(len(asr_df)),
            "speech_duration_sec": float((asr_df["end_time"] - asr_df["start_time"]).sum()),
            "word_count": int(sum(len(value) for value in asr_df["words"])),
            "avg_logprob_mean": float(asr_df["avg_logprob"].mean()),
            "low_confidence_segments": int((asr_df["avg_logprob"] < -1.0).sum()),
            "high_no_speech_segments": int((asr_df["no_speech_prob"] > 0.5).sum()),
        }
        (OUTPUT_ROOT / "asr-summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
        SUCCESS_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
        ERROR_PATH.unlink(missing_ok=True)

        archive = shutil.make_archive(str(BATCH_ROOT / f"04-asr-vietnamese-{VIDEO_ID}"), "zip", root_dir=OUTPUT_ROOT)
        success_count += 1
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        print("\nTranscript preview for quality check:")
        preview = asr_df[["start_time", "end_time", "text", "avg_logprob", "no_speech_prob"]].head(PREVIEW_SEGMENTS)
        display(preview)
        suspicious = asr_df.nsmallest(min(5, len(asr_df)), "avg_logprob")[["start_time", "end_time", "text", "avg_logprob"]]
        print("Lowest-confidence segments:")
        display(suspicious)
        print("Download:", archive)
    except Exception as exc:
        failure_count += 1
        ERROR_PATH.write_text(traceback.format_exc(), encoding="utf-8")
        failures.append({"video_id": VIDEO_ID, "error": f"{type(exc).__name__}: {exc}"})
        print(f"ERROR {VIDEO_ID}: {type(exc).__name__}: {exc}")
    finally:
        records = []
        globals().pop("asr_df", None)
        gc.collect()

batch_summary = {
    "videos_found": len(VIDEO_PATHS),
    "completed": success_count,
    "failed": failure_count,
    "failures": failures,
}
(BATCH_ROOT / "batch-summary.json").write_text(json.dumps(batch_summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(batch_summary, ensure_ascii=False, indent=2))
if failure_count:
    print("Some videos failed. Inspect _ERROR.txt; rerunning retries videos without _SUCCESS.json.")


## Output

Each video is saved under `/kaggle/working/04-asr-vietnamese-all-videos-output/<VIDEO_ID>/`.
